In [ ]:

import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
if str(root) not in sys.path:
    sys.path.append(str(root))

from src.data.transaction_utils import build_transactions
from src.mln.rule_utils import rules_to_mln_df

proc_dir = root / "data" / "processed" / "swat"
stream_dir = root / "data" / "stream" / "swat"
stream_dir.mkdir(parents=True, exist_ok=True)

print("proc_dir:", proc_dir)
print("stream_dir:", stream_dir)


In [ ]:

# 1) 读取二值特征与标签
df_binary = pd.read_csv(proc_dir / "X_filled_binary.csv")
y = pd.read_csv(proc_dir / "y_filled.csv").iloc[:, 0].astype(int).reset_index(drop=True)

lag = 2

print("df_binary.shape:", df_binary.shape)
print("y.shape:", y.shape)
print("label distribution:")
print(y.value_counts().sort_index())


In [ ]:

# 2) 生成全量 transactions，并与标签对齐
transactions = build_transactions(df_binary, lag=lag)
tx_labels = y.iloc[lag:].reset_index(drop=True)

assert len(transactions) == len(tx_labels), (len(transactions), len(tx_labels))

with open(stream_dir / "transactions_all.pkl", "wb") as f:
    pickle.dump(transactions, f)

tx_labels.to_csv(stream_dir / "transaction_labels.csv", index=False)

print("saved:", stream_dir / "transactions_all.pkl")
print("saved:", stream_dir / "transaction_labels.csv")
print("num_transactions:", len(transactions))
print("first_transaction:", transactions[0][:10] if len(transactions) > 0 else [])
print("tx_labels distribution:")
print(tx_labels.value_counts().sort_index())


In [ ]:

# 3) 保存前 2000 条 sample，便于快速检查
transactions_sample = transactions[:2000]

with open(stream_dir / "transactions_sample_2000.pkl", "wb") as f:
    pickle.dump(transactions_sample, f)

te = TransactionEncoder()
arr = te.fit(transactions_sample).transform(transactions_sample)
df_tx = pd.DataFrame(arr, columns=te.columns_)
df_tx.to_csv(stream_dir / "transactions_sample_2000_onehot.csv", index=False)

print("saved:", stream_dir / "transactions_sample_2000.pkl")
print("saved:", stream_dir / "transactions_sample_2000_onehot.csv")
print("sample 数量:", len(transactions_sample))
print("sample one-hot shape:", df_tx.shape)


In [ ]:

# 4) 找到围绕首次攻击点的 2000 长度窗口
first_attack_idx = int(tx_labels[tx_labels == 1].index[0])

window_size = 2000
left_normal_context = 1000

start = max(0, first_attack_idx - left_normal_context)
end = min(len(transactions), start + window_size)
start = max(0, end - window_size)  # 确保窗口长度尽量固定为 2000

transactions_attack_window = transactions[start:end]
labels_attack_window = tx_labels.iloc[start:end].reset_index(drop=True)

assert len(transactions_attack_window) == len(labels_attack_window)

with open(stream_dir / "transactions_attack_window_2000.pkl", "wb") as f:
    pickle.dump(transactions_attack_window, f)

labels_attack_window.to_csv(stream_dir / "labels_attack_window_2000.csv", index=False)

print("first_attack_idx:", first_attack_idx)
print("window:", start, end)
print("window_len:", len(transactions_attack_window))
print("窗口标签分布:")
print(labels_attack_window.value_counts().sort_index())
print("saved:", stream_dir / "transactions_attack_window_2000.pkl")
print("saved:", stream_dir / "labels_attack_window_2000.csv")


In [ ]:

# 5) attack window one-hot
te = TransactionEncoder()
arr = te.fit(transactions_attack_window).transform(transactions_attack_window)
df_attack_tx = pd.DataFrame(arr, columns=te.columns_)

df_attack_tx.to_csv(stream_dir / "transactions_attack_window_2000_onehot.csv", index=False)

print("saved:", stream_dir / "transactions_attack_window_2000_onehot.csv")
print("attack window one-hot shape:", df_attack_tx.shape)
print("前10列:", df_attack_tx.columns[:10].tolist())
display(df_attack_tx.iloc[:3, :10])


In [ ]:

# 6) 基于 attack window 挖掘频繁项集
freq_items_attack = fpgrowth(
    df_attack_tx,
    min_support=0.5,
    use_colnames=True,
    max_len=2
).sort_values("support", ascending=False).reset_index(drop=True)

freq_items_attack.to_csv(
    stream_dir / "freq_items_attack_window_2000_minsup_05_maxlen2.csv",
    index=False
)

print("saved:", stream_dir / "freq_items_attack_window_2000_minsup_05_maxlen2.csv")
print("attack window 频繁项集数量:", len(freq_items_attack))
display(freq_items_attack.head(20))


In [ ]:

# 7) 基于 attack window 普通规则挖掘
rules = association_rules(
    freq_items_attack,
    metric="confidence",
    min_threshold=0.8
)

rules = rules[
    (rules["antecedents"].apply(len) == 1) &
    (rules["consequents"].apply(len) == 1)
].copy()

rules = rules.sort_values(
    ["confidence", "lift", "support"],
    ascending=False
).reset_index(drop=True)

rules["antecedent_str"] = rules["antecedents"].apply(lambda x: list(x)[0])
rules["consequent_str"] = rules["consequents"].apply(lambda x: list(x)[0])

rules.to_csv(stream_dir / "rules_attack_window_2000_conf_08.csv", index=False)

print("saved:", stream_dir / "rules_attack_window_2000_conf_08.csv")
print("规则数量:", len(rules))
display(rules[["antecedent_str", "consequent_str", "support", "confidence", "lift"]].head(20))


In [ ]:

# 8) 转成初始 MLN 规则表
rules_df = pd.read_csv(stream_dir / "rules_attack_window_2000_conf_08.csv")
rules_df = rules_df.drop_duplicates(subset=["antecedent_str", "consequent_str"]).head(200).copy()

mln_rules_df = rules_to_mln_df(rules_df)
mln_rules_df.to_csv(stream_dir / "mln_rules_init_top200.csv", index=False)

print("saved:", stream_dir / "mln_rules_init_top200.csv")
print("MLN规则数量:", len(mln_rules_df))
display(mln_rules_df.head(10))


In [ ]:

# 9) 构造带标签 transactions
transactions_labeled = []
for tx, y_i in zip(transactions_attack_window, labels_attack_window.tolist()):
    tx_new = list(tx) + [("LABEL_ATTACK" if int(y_i) == 1 else "LABEL_NORMAL")]
    transactions_labeled.append(tx_new)

with open(stream_dir / "transactions_attack_window_2000_labeled.pkl", "wb") as f:
    pickle.dump(transactions_labeled, f)

print("saved:", stream_dir / "transactions_attack_window_2000_labeled.pkl")
print("labeled transactions 数量:", len(transactions_labeled))
print("第1条末尾:", transactions_labeled[0][-5:])
print("标签分布:")
print(pd.Series(labels_attack_window).value_counts().sort_index())


In [ ]:

# 10) labeled one-hot
te = TransactionEncoder()
arr = te.fit(transactions_labeled).transform(transactions_labeled)
df_labeled = pd.DataFrame(arr, columns=te.columns_)

df_labeled.to_csv(stream_dir / "transactions_attack_window_2000_labeled_onehot.csv", index=False)

print("saved:", stream_dir / "transactions_attack_window_2000_labeled_onehot.csv")
print("labeled one-hot shape:", df_labeled.shape)
print("最后几列:", df_labeled.columns[-5:].tolist())


In [ ]:

# 11) 挖掘到 LABEL_ATTACK / LABEL_NORMAL 的判别规则
freq_items_lbl = fpgrowth(
    df_labeled,
    min_support=0.1,
    use_colnames=True,
    max_len=2
)

rules_lbl = association_rules(
    freq_items_lbl,
    metric="confidence",
    min_threshold=0.6
)

rules_attack = rules_lbl[
    (rules_lbl["antecedents"].apply(len) == 1) &
    (rules_lbl["consequents"].apply(lambda x: x == frozenset({"LABEL_ATTACK"})))
].copy()

rules_attack["antecedent_str"] = rules_attack["antecedents"].apply(lambda x: list(x)[0])
rules_attack = rules_attack.sort_values(
    ["confidence", "lift", "support"],
    ascending=False
).reset_index(drop=True)

rules_normal = rules_lbl[
    (rules_lbl["antecedents"].apply(len) == 1) &
    (rules_lbl["consequents"].apply(lambda x: x == frozenset({"LABEL_NORMAL"})))
].copy()

rules_normal["antecedent_str"] = rules_normal["antecedents"].apply(lambda x: list(x)[0])
rules_normal = rules_normal.sort_values(
    ["confidence", "lift", "support"],
    ascending=False
).reset_index(drop=True)

rules_attack.to_csv(stream_dir / "rules_to_label_attack.csv", index=False)
rules_normal.to_csv(stream_dir / "rules_to_label_normal.csv", index=False)

print("saved:", stream_dir / "rules_to_label_attack.csv")
print("saved:", stream_dir / "rules_to_label_normal.csv")
print("攻击判别规则数:", len(rules_attack))
print("正常判别规则数:", len(rules_normal))
display(rules_attack[["antecedent_str", "support", "confidence", "lift"]].head(20))
display(rules_normal[["antecedent_str", "support", "confidence", "lift"]].head(20))


In [ ]:

# 12) 构造 attack rule pool 和 mixed rule pool
def confidence_to_weight(conf, eps=1e-6):
    conf = np.clip(conf, eps, 1 - eps)
    return float(np.log(conf / (1 - conf)))

attack_top = rules_attack.head(min(30, len(rules_attack))).copy()
attack_top["consequent_str"] = "LABEL_ATTACK"
attack_top["formula"] = attack_top["antecedent_str"] + " => LABEL_ATTACK"
attack_top["weight"] = attack_top["confidence"].apply(confidence_to_weight)
attack_top["target_label"] = 1

attack_rule_pool = attack_top[[
    "antecedent_str", "consequent_str",
    "formula", "support", "confidence", "lift", "weight", "target_label"
]].copy()

attack_rule_pool.to_csv(stream_dir / "mln_attack_rule_pool_top30.csv", index=False)

normal_top = rules_normal.head(min(9, len(rules_normal))).copy()
if len(normal_top) > 0:
    normal_top["consequent_str"] = "LABEL_NORMAL"
    normal_top["formula"] = normal_top["antecedent_str"] + " => LABEL_NORMAL"
    normal_top["weight"] = normal_top["confidence"].apply(confidence_to_weight)
    normal_top["target_label"] = 0

    mixed_rule_pool = pd.concat([
        attack_top.head(min(20, len(attack_top)))[["antecedent_str", "consequent_str", "formula", "support", "confidence", "lift", "weight", "target_label"]],
        normal_top[["antecedent_str", "consequent_str", "formula", "support", "confidence", "lift", "weight", "target_label"]],
    ], axis=0, ignore_index=True)
else:
    mixed_rule_pool = attack_top.head(min(20, len(attack_top)))[[
        "antecedent_str", "consequent_str", "formula", "support", "confidence", "lift", "weight", "target_label"
    ]].copy()

mixed_rule_pool.to_csv(stream_dir / "mln_mixed_rule_pool.csv", index=False)

print("saved:", stream_dir / "mln_attack_rule_pool_top30.csv")
print("saved:", stream_dir / "mln_mixed_rule_pool.csv")
print("攻击导向规则池数量:", len(attack_rule_pool))
print("混合规则池数量:", len(mixed_rule_pool))
print("mixed_rule_pool 标签分布:")
print(mixed_rule_pool["target_label"].value_counts())
display(mixed_rule_pool.head(10))


In [ ]:

# 13) 关键文件检查
key_files = [
    "transactions_all.pkl",
    "transactions_sample_2000.pkl",
    "transactions_sample_2000_onehot.csv",
    "transaction_labels.csv",
    "transactions_attack_window_2000.pkl",
    "labels_attack_window_2000.csv",
    "transactions_attack_window_2000_onehot.csv",
    "freq_items_attack_window_2000_minsup_05_maxlen2.csv",
    "rules_attack_window_2000_conf_08.csv",
    "mln_rules_init_top200.csv",
    "transactions_attack_window_2000_labeled.pkl",
    "transactions_attack_window_2000_labeled_onehot.csv",
    "rules_to_label_attack.csv",
    "rules_to_label_normal.csv",
    "mln_attack_rule_pool_top30.csv",
    "mln_mixed_rule_pool.csv",
]

for name in key_files:
    path = stream_dir / name
    print(name, "->", path.exists())
